# Compute metrics for Montserrat events, including creating SDS dayfiles if needed

In [ ]:
import os
from importlib import reload
import pandas as pd
import sys
import obspy
sys.path.append('lib')
import dataframe_tools as DFT
import pipelines
reload(pipelines)
reload(DFT)
paths = {}
paths['DATA_DIR'] = os.path.join('/data')
paths['SDS_DIR'] = os.path.join(paths['DATA_DIR'], 'SDS')
paths['SAM_DIR'] = os.path.join(paths['DATA_DIR'], 'SAM')
paths['RESPONSE_DIR'] = os.path.join('data', 'responses')
print(paths)



MLcsvfile = '/home/thompsong/Developer/kitchensinkGT/PROJECTS/MVOcatalog/to_dataframes/mvo_catalog_stats.csv'
MLcat = pd.read_csv(MLcsvfile)
R = 5000 # approx distance of MBWH from "dome" source
MLcat['MLamp'] = DFT.local_magnitude(MLcat['peakamp']*1e-6, R)
MLcat['MLA'] = DFT.local_magnitude(MLcat['peakA']*1e-6, R)
MLcat['ME'] = DFT.energy_magnitude(MLcat['energy'])

In [ ]:
MLcat.plot.scatter(x='MLA', y='ME')

In [ ]:
# Montserrat data from Seisan archive

## SCAFFOLD: THIS DOES NOT WORK PROPOERLY AS IT REPEATING EVENTS
seisandbdir =  '/data/SEISAN_DB/WAV/DSNC_'
net = 'MV'
invfile = os.path.join(paths['RESPONSE_DIR'],f"{net}.xml")
source = {'lat':16.71111, 'lon':-62.17722}
Ntry = 100
Nmin = 30
sampling_interval=2.56
catResultsDF = {}

#display(catResultsDF) 

subclasses = ['r', 'e', 'l', 'h', 't']
#subclasses = ['h', 't']
for subclass in subclasses:
    cat = MLcat[MLcat['subclass']==subclass]
    catResultsDF[subclass]=pd.DataFrame(columns=['Event', 'start', 'end', 'duration', 'ML', 'sum(ER)', 'ME', 'DR', 'DRS'])
    big = cat.nlargest(Ntry, 'ME')
    print(big)
    for i, row in big.iterrows():
        #print('\n', i, row, '\n')

        eventname = f"{subclass}_{row['filetime']}"
        startt = obspy.UTCDateTime(row['filetime'])
        endt = max([startt + min([row['trigger_duration'], 20]), obspy.UTCDateTime(row['offtime']) ] )
        print(f'Calling for data from {startt} to {endt}')

        # need to check SDS directory exists for this date, if not, we run the sausage to create it
        do_metric = {'SDS_RAW':True}   
        pipelines.big_sausage(seisandbdir, paths, startt, endt, \
                        sampling_interval=sampling_interval, \
                        source=source, \
                        invfile=invfile, \
                        Q=None, \
                        ext='pickle', \
                        dbout=None, \
                        net=net, do_metric=do_metric, MBWHZ_only=True)  
        
        print('Calling wrapper')
        DFT.wrapper(paths['SDS_DIR'], net, startt, endt, invfile, catResultsDF[subclass], eventname, source, sampling_interval=sampling_interval)
        if len(catResultsDF[subclass])==Nmin:
            break
        print('\n\n')
    print('\n\n****************')


print('\n\nDone\n')

In [ ]:
for subclass in subclasses:
    display(catResultsDF[subclass])
    print(catResultsDF[subclass][['ML', 'ME']].describe())
    DFT.fix_slope(catResultsDF[subclass], 'ML', 'ME', mfixed=4/3, cfixed=None, plot=True, print_stats=True)
    DFT.fix_slope(catResultsDF[subclass], 'ME', 'ML', mfixed=3/4, cfixed=None, plot=True, print_stats=True)